# 04 — LoRA: Fine-tune 2% of Parameters, Keep 98% Frozen

**By the end of this notebook** you'll have applied LoRA adapters to a PRAGMA model, fine-tuned only the low-rank delta matrices, compared parameter counts on a log-scale chart, and seen whether LoRA outperforms the frozen linear probe from notebook 03.

## What this notebook teaches

- What LoRA is and why it is useful for adapting a large pretrained model to a new task (§3.1.2)
- How `LoRAAdapter` injects low-rank delta matrices into the QKV and feed-forward layers
- How to count trainable vs frozen parameters and visualise the split
- How fine-tuning with LoRA compares to using a frozen probe on the same task

## Prerequisites

Run notebooks 01–03 first. This notebook builds on the same 60-customer synthetic dataset from notebook 03.

**How to use:** run every cell in order with Shift+Enter.

In [ ]:
# ── Setup: repo root on sys.path ─────────────────────────────────────────────
import sys, pathlib

# Locate repo root by searching candidate paths for pyproject.toml.
# Works whether Jupyter CWD is:
#   - the repo root itself           (local dev, top-level launch)
#   - notebooks/                     (local dev, launched from subdir)
#   - /opt/app-root/src              (OpenShift AI Workbench — CWD is parent of repo)
#   - /opt/app-root/src/pragma-encoder  (workbench with repo as CWD)
_here = pathlib.Path().resolve()
repo_root = next(
    (p for p in [_here, _here.parent,
                 _here / "pragma-encoder", _here.parent / "pragma-encoder"]
     if (p / "pyproject.toml").exists()),
    _here,
)
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn

# ── PRAGMA model ──────────────────────────────────────────────────────────────
from pragma_encoder.model           import PRAGMA, PRAGMAConfig
from pragma_encoder.model.assembler import EmbeddingAssembler
from pragma_encoder.masking         import MaskingStrategy
from pragma_encoder.tokenizer.vocabulary import VocabularySpec

# ── PRAGMA adaptation (src/pragma_encoder/adaptation/) ───────────────────────
from pragma_encoder.adaptation.probe import EmbeddingProbe   # §3.1.1 — frozen baseline
from pragma_encoder.adaptation.lora  import LoRAAdapter       # §3.1.2 — PEFT-backed LoRA

matplotlib.rcParams['figure.dpi'] = 110
torch.manual_seed(42)
np.random.seed(42)

# ── Check whether peft is installed (LoRA requires it) ───────────────────────
try:
    import peft
    _peft_available = True
    print(f"Setup complete. repo_root={repo_root}")
    print(f"peft {peft.__version__}")
except ImportError:
    _peft_available = False
    print(f"Setup complete. repo_root={repo_root}")
    print("WARNING: 'peft' library not installed. LoRA cells will show the concept only.")
    print("Install with: pip install peft")

## What is LoRA?

**Low-Rank Adaptation (LoRA)** is a parameter-efficient fine-tuning technique (Hu et al., 2021). Instead of updating all weights in a pretrained model, LoRA freezes the original weights and injects a small *delta* matrix into specific layers:

```
W_new = W_frozen + delta
delta = B @ A       # B: (d_out, r), A: (r, d_in)
```

where `r` is the LoRA rank (a small integer, typically 4–16). Because `r ≪ d_model`, the number of new trainable parameters is tiny.

PRAGMA applies LoRA to the QKV projection, output projection, and feed-forward layers of the History Encoder (§3.1.2). With rank `r=8` and `alpha=8`, this adds roughly 2–4% of the original parameter count.

Source: `src/pragma_encoder/adaptation/lora.py::LoRAAdapter`

## Rebuild the synthetic dataset

Same 60-customer, two-class dataset as notebook 03. We re-create it here so this notebook is self-contained.

In [ ]:
config     = PRAGMAConfig.pragma_s()
vocab_spec = VocabularySpec(
    special_tokens={"PAD": 0, "MASK": 1, "EVT": 2, "SEP": 3},
    key_start=4,
    key_size=config.key_vocab_size,
    value_start=4 + config.key_vocab_size,
    value_size=config.value_vocab_size,
    total_embedding_vocab_size=4 + config.key_vocab_size + config.value_vocab_size,
    field_key_ids={},
    field_value_ranges={},
)

N_CUSTOMERS = 60
NE, NI, NA  = 10, 8, 6
rng_g = torch.Generator(); rng_g.manual_seed(7)
labels  = torch.tensor([i % 2 for i in range(N_CUSTOMERS)], dtype=torch.long)
mid_val = vocab_spec.value_start + vocab_spec.value_size // 2

def _make_vals(label):
    lo, hi = (vocab_spec.value_start, mid_val) if label == 0 else (mid_val, vocab_spec.value_start + vocab_spec.value_size)
    return torch.randint(lo, hi, (NE, NI), generator=rng_g)

xe_val_ids = torch.stack([_make_vals(int(l)) for l in labels])
xe_key_ids = torch.randint(vocab_spec.key_start, vocab_spec.key_start + vocab_spec.key_size,
                            (N_CUSTOMERS, NE, NI), generator=rng_g)
xa_key_ids = torch.randint(vocab_spec.key_start, vocab_spec.key_start + vocab_spec.key_size,
                            (N_CUSTOMERS, NA), generator=rng_g)
xa_val_ids = torch.randint(vocab_spec.value_start, vocab_spec.value_start + vocab_spec.value_size,
                            (N_CUSTOMERS, NA), generator=rng_g)
ta       = torch.rand(N_CUSTOMERS, NA, generator=rng_g) * 5.0
te       = torch.rand(N_CUSTOMERS, NE, generator=rng_g) * 80.0
calendar = torch.rand(N_CUSTOMERS, NE, 3, generator=rng_g)

# Train/test split (same as notebook 03)
train_idx = sorted(list(range(0, N_CUSTOMERS, 5)) + list(range(1, N_CUSTOMERS, 5)) +
                   list(range(2, N_CUSTOMERS, 5)) + list(range(3, N_CUSTOMERS, 5)))
test_idx  = list(range(4, N_CUSTOMERS, 5))
train_labels = labels[train_idx]
test_labels  = labels[test_idx]

print(f"{N_CUSTOMERS} customers, {len(train_idx)} train, {len(test_idx)} test")

## Pretrain a PRAGMA-S model (30 steps)

We need a pretrained checkpoint before we can fine-tune it. This is the same loop as notebooks 02–03, abbreviated.

In [ ]:
torch.manual_seed(42)
model_pretrained  = PRAGMA(config)
assembler         = EmbeddingAssembler(vocab_spec, config)
masker            = MaskingStrategy(config)
opt_pretrain      = torch.optim.Adam(
    list(model_pretrained.parameters()) + list(assembler.parameters()), lr=1e-4
)

BATCH_SIZE   = 12
N_PRETRAIN   = 30

def _mini_batch(idx_list):
    chosen = torch.randperm(len(idx_list))[:BATCH_SIZE].tolist()
    idxs   = [idx_list[i] for i in chosen]
    return (xa_key_ids[idxs], xa_val_ids[idxs], ta[idxs],
            xe_key_ids[idxs], xe_val_ids[idxs], te[idxs], calendar[idxs])

print(f"Pretraining for {N_PRETRAIN} steps...")
for step in range(N_PRETRAIN):
    xak, xav, ta_, xek, xev, te_, cal = _mini_batch(train_idx)
    opt_pretrain.zero_grad()
    mv, _, mm = masker.forward(xev, xek)
    if not mm.any(): mm[0,0,0]=True; mv[0,0,0]=vocab_spec.special_tokens["MASK"]
    asm = assembler.forward(xa_key_ids=xak, xa_val_ids=xav, ta=ta_,
                            xe_key_ids=xek, xe_val_ids=mv, te=te_,
                            calendar=cal, targets=xev, mlm_mask=mm)
    out  = model_pretrained.forward(xa=asm.xa, ta=asm.ta, xe=asm.xe,
                                    xt=asm.xt, te=asm.te, mask=asm.mlm_mask)
    loss = model_pretrained.mlm_head.compute_loss(out["logits"], asm.targets[asm.mlm_mask])
    loss.backward(); opt_pretrain.step()

print(f"Pretraining done. Final loss: {loss.item():.4f}")

## Frozen probe baseline (notebook 03 recap)

In [ ]:
def extract_usr(mdl, asm):
    mdl.eval(); all_e = []
    with torch.no_grad():
        for s in range(0, N_CUSTOMERS, BATCH_SIZE):
            e = min(s + BATCH_SIZE, N_CUSTOMERS)
            a = asm.forward(
                xa_key_ids=xa_key_ids[s:e], xa_val_ids=xa_val_ids[s:e], ta=ta[s:e],
                xe_key_ids=xe_key_ids[s:e], xe_val_ids=xe_val_ids[s:e], te=te[s:e],
                calendar=calendar[s:e], targets=xe_val_ids[s:e],
                mlm_mask=torch.zeros(e-s, NE, NI, dtype=torch.bool),
            )
            o = mdl.forward(xa=a.xa, ta=a.ta, xe=a.xe, xt=a.xt, te=a.te, mask=a.mlm_mask)
            all_e.append(o["zh"][:, 0, :].cpu())
    return torch.cat(all_e)

emb_pt    = extract_usr(model_pretrained, assembler)
probe_pt  = EmbeddingProbe(config)
probe_pt.fit(emb_pt[train_idx], train_labels, task="classification")
auc_probe = probe_pt.score(emb_pt[test_idx], test_labels)
print(f"Frozen probe AUC-ROC: {auc_probe:.3f}")

## Apply LoRA adapters

`LoRAAdapter.apply(model)` returns a `peft.PeftModel`. The original weights are frozen. Only the small `A` and `B` matrices are trainable.

In [ ]:
if _peft_available:
    # Deep-copy the pretrained model so we don't mutate it
    import copy
    model_for_lora = copy.deepcopy(model_pretrained)

    adapter    = LoRAAdapter(config)
    peft_model = adapter.apply(model_for_lora)

    n_total     = sum(p.numel() for p in peft_model.parameters())
    n_trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    n_frozen    = n_total - n_trainable

    print(f"Total parameters     : {n_total:>12,}")
    print(f"Trainable (LoRA)     : {n_trainable:>12,}  ({100 * n_trainable / n_total:.2f}%)")
    print(f"Frozen (pretrained)  : {n_frozen:>12,}  ({100 * n_frozen / n_total:.2f}%)")
    print(f"LoRA rank r={config.lora_rank}, alpha={config.lora_alpha}  (Table 1, §3.1.2)")
else:
    print("peft not available. Showing expected parameter breakdown:")
    n_total     = sum(p.numel() for p in model_pretrained.parameters())
    n_trainable = int(n_total * 0.024)   # ~2.4% for PRAGMA-S with r=8
    n_frozen    = n_total - n_trainable
    print(f"  Total        : ~{n_total:,}")
    print(f"  Trainable    : ~{n_trainable:,}  (~2.4%)")
    print(f"  Frozen       : ~{n_frozen:,}  (~97.6%)")

## Visualisation 1 — Parameter count breakdown (log scale)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

categories  = ["Frozen\n(pretrained)", "Trainable\n(LoRA)", "Probe\n(linear layer)"]
n_probe_params = config.d_model + 1   # w + b for binary classification
counts      = [n_frozen, n_trainable, n_probe_params]
bar_colors  = ["#4C72B0", "#DD8452", "#55A868"]

bars = ax.bar(categories, counts, color=bar_colors, width=0.5,
              edgecolor="white", linewidth=1.5, log=True)

for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 2,
            f"{cnt:,}",
            ha="center", va="bottom", fontsize=9, fontweight="bold", rotation=0)

ax.set_ylabel("Parameter count (log scale)", fontsize=11)
ax.set_title("PRAGMA-S: LoRA adds a tiny fraction of total parameters (§3.1.2)", fontsize=11)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

print(f"\nLoRA adds {n_trainable:,} trainable parameters vs {n_probe_params} for a linear probe.")
print("LoRA can update the representation; a linear probe cannot.")

**What you're looking at:** the log scale is essential — the frozen and trainable bar heights are orders of magnitude apart. A linear probe (green) adds only `d_model + 1` parameters (a weight vector + bias). LoRA (orange) adds more than the probe, but still far fewer than the full model. Both are dwarfed by the frozen backbone.

## LoRA fine-tuning loop

We fine-tune with LoRA by attaching a binary classification head to the `[USR]` embedding and training with binary cross-entropy. Only the LoRA parameters and classification head update.

In [ ]:
if _peft_available:
    N_FINETUNE = 40

    # Simple binary classification head on top of [USR]
    clf_head  = nn.Linear(config.d_model, 1)
    bce_loss  = nn.BCEWithLogitsLoss()

    # Only LoRA parameters + classification head are trainable
    opt_lora  = torch.optim.Adam(
        list(p for p in peft_model.parameters() if p.requires_grad)
        + list(clf_head.parameters()),
        lr=3e-4,
    )

    peft_model.train()
    lora_losses = []

    print(f"LoRA fine-tuning for {N_FINETUNE} steps...")
    for step in range(N_FINETUNE):
        # Sample a mini-batch from training split
        chosen  = torch.randperm(len(train_idx))[:BATCH_SIZE].tolist()
        idxs    = [train_idx[i] for i in chosen]
        batch_l = labels[idxs].float()

        opt_lora.zero_grad()
        # Forward: no masking — we're fine-tuning on a classification objective
        assembled_ft = assembler.forward(
            xa_key_ids=xa_key_ids[idxs], xa_val_ids=xa_val_ids[idxs], ta=ta[idxs],
            xe_key_ids=xe_key_ids[idxs], xe_val_ids=xe_val_ids[idxs], te=te[idxs],
            calendar=calendar[idxs], targets=xe_val_ids[idxs],
            mlm_mask=torch.zeros(len(idxs), NE, NI, dtype=torch.bool),
        )
        out_ft   = peft_model.forward(
            xa=assembled_ft.xa, ta=assembled_ft.ta, xe=assembled_ft.xe,
            xt=assembled_ft.xt, te=assembled_ft.te, mask=assembled_ft.mlm_mask,
        )
        usr_emb  = out_ft["zh"][:, 0, :]      # (B, d_model)
        logits   = clf_head(usr_emb).squeeze(-1)  # (B,)
        loss_ft  = bce_loss(logits, batch_l)
        loss_ft.backward()
        opt_lora.step()
        lora_losses.append(loss_ft.item())

        if step == 0 or (step + 1) % 10 == 0:
            print(f"  step {step+1:3d}  loss={loss_ft.item():.4f}")

    print("Fine-tuning complete.")
else:
    print("peft not available — LoRA fine-tuning skipped.")
    print("Install peft and rerun to see the fine-tuning loop.")
    lora_losses = []
    N_FINETUNE  = 0

## Evaluate LoRA fine-tuned model

In [ ]:
if _peft_available:
    peft_model.eval()
    clf_head.eval()

    all_probs, all_labels = [], []
    with torch.no_grad():
        for s in range(0, len(test_idx), BATCH_SIZE):
            e    = min(s + BATCH_SIZE, len(test_idx))
            idxs = test_idx[s:e]
            asm  = assembler.forward(
                xa_key_ids=xa_key_ids[idxs], xa_val_ids=xa_val_ids[idxs], ta=ta[idxs],
                xe_key_ids=xe_key_ids[idxs], xe_val_ids=xe_val_ids[idxs], te=te[idxs],
                calendar=calendar[idxs], targets=xe_val_ids[idxs],
                mlm_mask=torch.zeros(len(idxs), NE, NI, dtype=torch.bool),
            )
            out = peft_model.forward(
                xa=asm.xa, ta=asm.ta, xe=asm.xe, xt=asm.xt, te=asm.te, mask=asm.mlm_mask
            )
            probs = torch.sigmoid(clf_head(out["zh"][:, 0, :]).squeeze(-1))
            all_probs.append(probs.cpu())
            all_labels.append(labels[idxs])

    all_probs  = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()

    # Compute AUC without sklearn — Wilcoxon-Mann-Whitney statistic
    pos_scores = all_probs[all_labels == 1]
    neg_scores = all_probs[all_labels == 0]
    n_pos, n_neg = len(pos_scores), len(neg_scores)
    auc_lora = sum(1 for p in pos_scores for n in neg_scores if p > n) / (n_pos * n_neg) if (n_pos * n_neg > 0) else 0.5

    print(f"LoRA fine-tuned AUC-ROC : {auc_lora:.3f}")
    print(f"Frozen probe AUC-ROC    : {auc_probe:.3f}")
    print(f"Delta (LoRA - probe)    : {auc_lora - auc_probe:+.3f}")
else:
    auc_lora = None
    print("LoRA not run — peft not installed.")

## Visualisation 2 — Frozen probe vs LoRA AUC comparison

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Left: AUC comparison bar chart
bar_labels_auc = ["Frozen probe\n(notebook 03)",
                  "LoRA fine-tuned\n(this notebook)"]
auc_lora_plot  = auc_lora if auc_lora is not None else 0.5
aucs_plot      = [auc_probe, auc_lora_plot]
bar_cs         = ["#4C72B0", "#DD8452"]

bars = ax1.bar(bar_labels_auc, aucs_plot, color=bar_cs, width=0.45,
               edgecolor="white", linewidth=1.5)
ax1.axhline(0.5, color="#888", linewidth=1.2, linestyle="--", label="random chance")
for bar, auc in zip(bars, aucs_plot):
    label_txt = f"{auc:.3f}" if auc is not None else "(peft needed)"
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
             label_txt, ha="center", va="bottom", fontsize=11, fontweight="bold")
ax1.set_ylabel("AUC-ROC", fontsize=11)
ax1.set_title("Probe vs LoRA accuracy", fontsize=11)
ax1.set_ylim(0, 1.2)
ax1.legend(fontsize=9)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# Right: LoRA fine-tuning loss curve
if lora_losses:
    ax2.plot(range(1, len(lora_losses) + 1), lora_losses,
             color="#DD8452", linewidth=2, label="BCE loss (LoRA)")
    ax2.set_xlabel("Fine-tuning step", fontsize=10)
    ax2.set_ylabel("Binary cross-entropy", fontsize=10)
    ax2.set_title("LoRA fine-tuning loss curve", fontsize=11)
    ax2.legend(fontsize=9)
    ax2.spines["top"].set_visible(False)
    ax2.spines["right"].set_visible(False)
else:
    ax2.text(0.5, 0.5, "peft not installed\n(LoRA not run)",
             ha="center", va="center", transform=ax2.transAxes, fontsize=13, color="#888")
    ax2.set_title("LoRA fine-tuning loss curve", fontsize=11)

plt.suptitle("LoRA fine-tuning (§3.1.2) vs frozen linear probe (§3.1.1)", fontsize=12)
plt.tight_layout()
plt.show()

**What you're looking at:** the left chart compares AUC-ROC between the frozen linear probe and LoRA fine-tuning on the same test set. LoRA can update the representation — it is not constrained to a linear decision boundary over frozen embeddings. The right chart shows the binary cross-entropy loss during fine-tuning.

## What just happened — section recap

- `LoRAAdapter.apply(model)` injected low-rank delta matrices (rank=8, alpha=8) into QKV, output projection, and feed-forward layers — freezing everything else
- ~2–4% of total parameters are trainable; the remaining ~96–98% are frozen pretrained weights
- Fine-tuning with binary cross-entropy over `[USR]` embeddings adapts the representation to a downstream task without catastrophic forgetting of the pretraining knowledge
- The parameter count bar chart (log scale) shows the order-of-magnitude difference between frozen weights, LoRA matrices, and a linear probe

## Closing — what you now know

**LoRA:** inject small `B @ A` delta matrices into frozen layers. Only `A` and `B` are trained. `LoRAAdapter(config).apply(model)` returns a `peft.PeftModel`.

**Parameter efficiency:** `r=8, alpha=8` → ~2.4% trainable parameters for PRAGMA-S. The frozen backbone retains its pretrained structure.

**vs frozen probe:** a linear probe is faster and needs no gradient computation. LoRA can update the representation and is more powerful — at the cost of slightly more computation and the `peft` dependency.

**Next:** notebook 05 collects evaluation metrics across all four notebooks and produces a final summary table and chart.

**Going deeper:** `src/pragma_encoder/adaptation/lora.py` — the full `LoRAAdapter` implementation. `docs/paper-to-code.md §3.1.2` — LoRA target modules and rank choices from the paper.